# 03 — Líneas base y modelos candidatos

Se reserva una sola prueba por grupos de compuesto. La comparación y el tuning usan únicamente entrenamiento y el mismo GroupKFold de cinco pliegues. La mediana por condición es un proxy basado en datos históricos, no una medición del proceso operativo actual del laboratorio.

In [1]:
from pathlib import Path
import os
import sys
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.datos import load_snapshot
from src.experiments import evaluate_cv_run
from src.pipeline import (
    make_baselines,
    make_default_models,
    make_group_cv,
    make_model_pipeline,
    model_inputs,
    valid_smiles_mask,
)
from src.splits import create_or_load_holdout, stratified_compound_sample

frame, metadata = load_snapshot()
valid = valid_smiles_mask(frame)
all_data = frame.loc[valid & frame["inchikey"].notna()].reset_index(drop=True)
train_full, holdout_full, split_metadata = create_or_load_holdout(all_data, source_sha256=metadata["sha256"])
train_sample, sample_metadata = stratified_compound_sample(train_full, max_compounds=12_000, random_state=42)
model_metadata = {**metadata, "source_rows": metadata["rows"], "rows": len(train_sample), "training_sample_sha256": sample_metadata["sha256"], "sampling_method": sample_metadata["method"], "sampling_seed": sample_metadata["random_state"]}
X_train, y_train, groups_train = model_inputs(train_sample)
cv = make_group_cv(n_splits=5)
print({"filas_totales": len(all_data), "train_completo": len(train_full), "holdout_completo_reservado": len(holdout_full), "muestra_seleccion": len(train_sample), "compuestos_muestra": sample_metadata["compounds"], "grupos_train_sample": groups_train.nunique(), "pliegues": cv.get_n_splits()})

{'filas_totales': 152586, 'train_completo': 122370, 'holdout_completo_reservado': 30216, 'muestra_seleccion': 19406, 'compuestos_muestra': 12000, 'grupos_train_sample': 12000, 'pliegues': 5}


In [2]:
baseline_results = []
for family, estimator in make_baselines().items():
    baseline_results.append(evaluate_cv_run(
        run_name=f"baseline_{family}", phase="baseline", family=family,
        estimator=estimator, X=X_train, y=y_train, groups=groups_train, cv=cv,
        dataset_metadata=model_metadata,
    ))
baseline_table = pd.DataFrame(baseline_results).sort_values("cv_rmse_mean")
display(baseline_table[["family", "train_rmse_mean", "cv_rmse_mean", "cv_rmse_std", "cv_mae_mean", "cv_fit_seconds_total"]])

candidate_results = []
for family, regressor in make_default_models().items():
    candidate_results.append(evaluate_cv_run(
        run_name=f"candidate_{family}", phase="candidate", family=family,
        estimator=make_model_pipeline(regressor), X=X_train, y=y_train,
        groups=groups_train, cv=cv, dataset_metadata=model_metadata,
    ))
candidate_table = pd.DataFrame(candidate_results).sort_values("cv_rmse_mean")
display(candidate_table[["family", "train_rmse_mean", "cv_rmse_mean", "cv_rmse_std", "rmse_train_cv_gap", "cv_mae_mean", "cv_fit_seconds_total"]])

docs = PROJECT_ROOT / "docs"
docs.mkdir(exist_ok=True)
baseline_table.to_csv(docs / "baselines.csv", index=False)
candidate_table.to_csv(docs / "candidatos.csv", index=False)
print(f"Resultados guardados en {docs}")

🏃 View run baseline_naive_median at: http://localhost:5000/#/experiments/1/runs/711f323b55de442dab08c0e9ab1a75d2
🧪 View experiment at: http://localhost:5000/#/experiments/1


🏃 View run baseline_method_median_proxy at: http://localhost:5000/#/experiments/1/runs/4e7367e53b4c42789de9d72ccec8efde
🧪 View experiment at: http://localhost:5000/#/experiments/1


,family,train_rmse_mean,cv_rmse_mean,cv_rmse_std,cv_mae_mean,cv_fit_seconds_total
1,method_median_proxy,3.691914,3.764737,0.404978,2.583680,0.162976
0,naive_median,5.826243,5.828258,0.090012,4.875349,0.092283


🏃 View run candidate_ridge at: http://localhost:5000/#/experiments/1/runs/6a5033ec73dd4bf1852e431228298a42
🧪 View experiment at: http://localhost:5000/#/experiments/1


🏃 View run candidate_random_forest at: http://localhost:5000/#/experiments/1/runs/52e464f377334b1aa40885a14923f91a
🧪 View experiment at: http://localhost:5000/#/experiments/1


🏃 View run candidate_hist_gradient_boosting at: http://localhost:5000/#/experiments/1/runs/d760877a10104c9fb9d325e9a8bb3baf
🧪 View experiment at: http://localhost:5000/#/experiments/1


... (6 outputs left out)

🏃 View run candidate_mlp at: http://localhost:5000/#/experiments/1/runs/49b56853cae941cab73beda95f653f06
🧪 View experiment at: http://localhost:5000/#/experiments/1


🏃 View run candidate_knn at: http://localhost:5000/#/experiments/1/runs/e7209eb441ac43388c705a6fb4a64886
🧪 View experiment at: http://localhost:5000/#/experiments/1


,family,train_rmse_mean,cv_rmse_mean,cv_rmse_std,rmse_train_cv_gap,cv_mae_mean,cv_fit_seconds_total
2,hist_gradient_boosting,2.132407,2.604109,0.239992,0.471702,1.693492,11.186555
1,random_forest,1.119416,2.682672,0.247504,1.563256,1.722910,41.687265
3,mlp,2.451082,2.764227,0.204090,0.313145,1.846046,66.093571
4,knn,2.089474,3.064172,0.211547,0.974698,1.934838,14.125754
0,ridge,3.154710,3.176234,0.199588,0.021524,2.278108,8.116531


Resultados guardados en /workspaces/data-science-end-to-end/psets/pset-03/docs


## LLM mediante prompting: alcance y reproducibilidad

Prompt de evaluación propuesto: *«Predice el tiempo de retención (min) para este SMILES y estas condiciones C18 (fase A/B, gradiente, flujo y temperatura). Devuelve solo un número y explica brevemente qué evidencia sustenta la predicción.»* Un LLM genérico no incorpora de forma fiable la química de retención ni el gradiente experimental; sin un modelo local/específico, API aprobada y versión congelada no se puede registrar una comparación reproducible. Por eso no se presentan respuestas generadas como resultados del benchmark ni se envían datos a un servicio externo. Esta limitación debe declararse en la entrega; si el profesor exige un resultado cuantitativo de LLM, hará falta acordar un endpoint/modelo autorizado antes de ejecutarlo.